
# SQL Interview Lab — Question 2
## Customer Deduplication & Incremental CDC Processing

### Business Problem
Customer records may contain multiple versions,
timestamp ties, and late-arriving updates.

Our goal is to maintain one correct current-state
record per customer without losing source history.

### Technologies
- Python
- pandas
- SQL / SQLite
- JupyterLab

### Concepts Implemented
1. CSV ingestion and landing tables
2. Source-to-target reconciliation
3. ROW_NUMBER() and window functions
4. CTE-based deduplication
5. Deterministic timestamp tie-breaking
6. Late-arriving data handling
7. Conditional UPSERT
8. Idempotency and safe replay
9. Data Quality validation
10. CDC audit and value-level reconciliation

### Validated Results
- Landing records: 4
- Distinct customers: 2
- Current-state customers: 2
- UPSERT event attempts: 4
- Inserts: 1
- Updates: 1
- Ignored attempts: 2
- Value-level matches: 2
- Value-level mismatches: 0

### Project Outcome
Implemented and validated customer deduplication
and incremental current-state processing using
synthetic banking customer data.


In [2]:

from pathlib import Path

# Create project folder
project = Path(".")
project.mkdir(exist_ok=True)

# Create sample customer records
csv_data = """record_id,customer_id,city,updated_at
101,C001,Hyderabad,2026-10-20
102,C001,Chennai,2026-10-22
103,C001,Bengaluru,2026-10-24
104,C002,Mumbai,2026-10-23
"""

# Save CSV file
csv_path = project / "customers.csv"
csv_path.write_text(csv_data)

# Verify source data
print("CSV file created successfully!")
print(csv_path.read_text())


CSV file created successfully!
record_id,customer_id,city,updated_at
101,C001,Hyderabad,2026-10-20
102,C001,Chennai,2026-10-22
103,C001,Bengaluru,2026-10-24
104,C002,Mumbai,2026-10-23



In [3]:

import pandas as pd

# Read the customer CSV
df = pd.read_csv(
    "customers.csv"
)

# Display all records
print("Customer source records:")
display(df)

# Check total records
print("Total records:", len(df))

# Inspect column data types
print("\nColumn data types:")
print(df.dtypes)


Customer source records:


,record_id,customer_id,city,updated_at
0,101,C001,Hyderabad,2026-10-20
1,102,C001,Chennai,2026-10-22
2,103,C001,Bengaluru,2026-10-24
3,104,C002,Mumbai,2026-10-23


Total records: 4

Column data types:
record_id       int64
customer_id    object
city           object
updated_at     object
dtype: object


In [4]:

import sqlite3

# Connect to our Question 2 SQLite database
conn = sqlite3.connect(
    "customers.db"
)

# Load pandas DataFrame into SQLite
df.to_sql(
    "customers_landing",
    conn,
    if_exists="replace",
    index=False
)

print("Customer landing table created successfully!")


Customer landing table created successfully!


In [5]:

# Step 4: Verify SQLite landing records

# Query 1: Read all customer records
query_records = """
SELECT *
FROM customers_landing
ORDER BY record_id;
"""

customers_result = pd.read_sql_query(
    query_records, conn
)

print("Customer records:")
display(customers_result)

# Query 2: Count records using SQL
query_count = """
SELECT COUNT(*) AS total_records
FROM customers_landing;
"""

count_result = pd.read_sql_query(
    query_count, conn
)

print("SQL count:")
display(count_result)

# Source-to-target count reconciliation
source_count = len(df)
target_count = count_result["total_records"].iloc[0]

print("Source records:", source_count)
print("Target records:", target_count)
print("Difference:", source_count - target_count)


Customer records:


,record_id,customer_id,city,updated_at
0,101,C001,Hyderabad,2026-10-20
1,102,C001,Chennai,2026-10-22
2,103,C001,Bengaluru,2026-10-24
3,104,C002,Mumbai,2026-10-23


SQL count:


,total_records
0,4


Source records: 4
Target records: 4
Difference: 0


In [6]:
#STEP 5 :Assign row numbers to customer versions

query_row_number = """
SELECT
      record_id,
      customer_id,
      city,
      updated_at,
      ROW_NUMBER()OVER(PARTITION BY customer_id
      ORDER BY updated_at DESC, record_id DESC) AS rn
FROM customers_landing
ORDER BY customer_id,rn;
"""
ranked_customers = pd.read_sql_query(
    query_row_number,
    conn
)

print("Customer records with ROW_NUMBER:")
display(ranked_customers)

Customer records with ROW_NUMBER:


,record_id,customer_id,city,updated_at,rn
0,103,C001,Bengaluru,2026-10-24,1
1,102,C001,Chennai,2026-10-22,2
2,101,C001,Hyderabad,2026-10-20,3
3,104,C002,Mumbai,2026-10-23,1


In [7]:

# Step 6: Select latest record per customer using a CTE

query_latest_customers = """
WITH ranked_customers AS (
    SELECT
        record_id,
        customer_id,
        city,
        updated_at,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY updated_at DESC, record_id DESC
        ) AS rn
    FROM customers_landing
)
SELECT
    record_id,
    customer_id,
    city,
    updated_at
FROM ranked_customers
WHERE rn = 1
ORDER BY customer_id;
"""

latest_customers_df = pd.read_sql_query(
    query_latest_customers,
    conn
)

print("Latest record for each customer:")
display(latest_customers_df)

print("Total latest customer records:",
      len(latest_customers_df))


Latest record for each customer:


,record_id,customer_id,city,updated_at
0,103,C001,Bengaluru,2026-10-24
1,104,C002,Mumbai,2026-10-23


Total latest customer records: 2


In [8]:

# Step 7: Validate deduplication results

validation_query = """
WITH ranked AS (
    SELECT
        record_id,
        customer_id,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY updated_at DESC, record_id DESC
        ) AS rn
    FROM customers_landing
),
latest AS (
    SELECT customer_id
    FROM ranked
    WHERE rn = 1
),
metrics AS (
    SELECT
        (SELECT COUNT(*) FROM customers_landing)
            AS landing_count,

        (SELECT COUNT(DISTINCT customer_id)
         FROM customers_landing)
            AS distinct_customers,

        (SELECT COUNT(*) FROM latest)
            AS latest_count,

        (SELECT COUNT(*) FROM ranked WHERE rn > 1)
            AS older_count,

        (SELECT COUNT(*) - COUNT(DISTINCT customer_id)
         FROM latest)
            AS duplicate_keys,

        (SELECT COUNT(*) FROM latest
         WHERE customer_id IS NULL)
            AS null_customer_keys
)
SELECT
    *,
    CASE
        WHEN landing_count = latest_count + older_count
         AND latest_count = distinct_customers
         AND duplicate_keys = 0
         AND null_customer_keys = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status
FROM metrics;
"""

validation_df = pd.read_sql_query(
    validation_query,
    conn
)

print("Deduplication validation results:")
display(validation_df)


Deduplication validation results:


,landing_count,distinct_customers,latest_count,older_count,duplicate_keys,null_customer_keys,validation_status
0,4,2,2,2,0,0,PASS


In [9]:

# Step 8: Timestamp ties and deterministic ranking

query_tie_breaker = """
WITH sample_updates AS (
    SELECT 105 AS record_id,
           'C001' AS customer_id,
           'Pune' AS city,
           '2026-10-25 10:00:00' AS updated_at

    UNION ALL

    SELECT 106,
           'C001',
           'Delhi',
           '2026-10-25 10:00:00'
),
ranked AS (
    SELECT
        record_id,
        customer_id,
        city,
        updated_at,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY updated_at DESC, record_id DESC
        ) AS rn
    FROM sample_updates
)
SELECT *
FROM ranked
ORDER BY rn;
"""

tie_result = pd.read_sql_query(
    query_tie_breaker,
    conn
)

print("Timestamp tie-breaker results:")
display(tie_result)


Timestamp tie-breaker results:


,record_id,customer_id,city,updated_at,rn
0,106,C001,Delhi,2026-10-25 10:00:00,1
1,105,C001,Pune,2026-10-25 10:00:00,2


In [10]:

# Step 9: Handle late-arriving customer updates

query_late_arrivals = """
WITH customer_events AS (
    SELECT
        201 AS record_id,
        'C001' AS customer_id,
        'Hyderabad' AS city,
        '2026-10-25 09:00:00' AS event_time,
        '2026-10-25 09:01:00' AS arrival_time

    UNION ALL

    SELECT
        202, 'C001', 'Bengaluru',
        '2026-10-25 11:00:00',
        '2026-10-25 11:02:00'

    UNION ALL

    SELECT
        203, 'C001', 'Chennai',
        '2026-10-25 10:00:00',
        '2026-10-25 12:00:00'
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY event_time DESC, record_id DESC
        ) AS rn_event,

        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY arrival_time DESC, record_id DESC
        ) AS rn_arrival

    FROM customer_events
)
SELECT
    record_id,
    customer_id,
    city,
    event_time,
    arrival_time,
    rn_event,
    rn_arrival
FROM ranked
ORDER BY record_id;
"""

late_arrival_df = pd.read_sql_query(
    query_late_arrivals,
    conn
)

print("Late-arriving customer event comparison:")
display(late_arrival_df)


Late-arriving customer event comparison:


,record_id,customer_id,city,event_time,arrival_time,rn_event,rn_arrival
0,201,C001,Hyderabad,2026-10-25 09:00:00,2026-10-25 09:01:00,3,3
1,202,C001,Bengaluru,2026-10-25 11:00:00,2026-10-25 11:02:00,1,2
2,203,C001,Chennai,2026-10-25 10:00:00,2026-10-25 12:00:00,2,1


In [11]:

# Step 10: Protect current state from late-arriving data

query_current_state = """
WITH current_state AS (
    SELECT
        'C001' AS customer_id,
        'Bengaluru' AS city,
        '2026-10-25 11:00:00' AS event_time,
        202 AS record_id
),
incoming_update AS (
    SELECT
        'C001' AS customer_id,
        'Chennai' AS city,
        '2026-10-25 10:00:00' AS event_time,
        203 AS record_id
),
comparison AS (
    SELECT
        c.customer_id,
        c.city AS existing_city,
        i.city AS incoming_city,
        c.event_time AS existing_time,
        i.event_time AS incoming_time,
        CASE
            WHEN i.event_time > c.event_time
              OR (
                  i.event_time = c.event_time
                  AND i.record_id > c.record_id
              )
            THEN 'APPLY_UPDATE'
            ELSE 'KEEP_EXISTING'
        END AS update_decision
    FROM current_state c
    JOIN incoming_update i
      ON c.customer_id = i.customer_id
)
SELECT *
FROM comparison;
"""

decision_df = pd.read_sql_query(
    query_current_state,
    conn
)

print("Late-arriving update decision:")
display(decision_df)


Late-arriving update decision:


,customer_id,existing_city,incoming_city,existing_time,incoming_time,update_decision
0,C001,Bengaluru,Chennai,2026-10-25 11:00:00,2026-10-25 10:00:00,KEEP_EXISTING


In [12]:

# Step 11A: Create a current-state customer table

conn.execute("""
CREATE TABLE IF NOT EXISTS customers_current (
    customer_id TEXT PRIMARY KEY,
    city TEXT NOT NULL,
    event_time TEXT NOT NULL,
    record_id INTEGER NOT NULL
);
""")

# Reset the lab table to a known starting state
# Do not use this reset in production.
conn.execute("DELETE FROM customers_current;")

# Insert the existing latest customer record
conn.execute("""
INSERT INTO customers_current (
    customer_id,
    city,
    event_time,
    record_id
)
VALUES (?, ?, ?, ?);
""", (
    "C001",
    "Bengaluru",
    "2026-10-25 11:00:00",
    202
))

# Save changes
conn.commit()

# Verify current state
current_df = pd.read_sql_query(
    "SELECT * FROM customers_current;",
    conn
)

print("Initial current-state table:")
display(current_df)


Initial current-state table:


,customer_id,city,event_time,record_id
0,C001,Bengaluru,2026-10-25 11:00:00,202


In [13]:

# Step 11B: Conditional UPSERT for an older event

upsert_sql = """
INSERT INTO customers_current (
    customer_id,
    city,
    event_time,
    record_id
)
VALUES (?, ?, ?, ?)

ON CONFLICT(customer_id)
DO UPDATE SET
    city = excluded.city,
    event_time = excluded.event_time,
    record_id = excluded.record_id

WHERE
    excluded.event_time > customers_current.event_time
    OR (
        excluded.event_time = customers_current.event_time
        AND excluded.record_id > customers_current.record_id
    );
"""

# Incoming late-arriving event
incoming_event = (
    "C001",
    "Chennai",
    "2026-10-25 10:00:00",
    203
)

# Execute the actual UPSERT
cursor = conn.execute(upsert_sql, incoming_event)

# Commit the transaction
conn.commit()

# Verify the database state
result_df = pd.read_sql_query(
    "SELECT * FROM customers_current;",
    conn
)

print("Rows inserted or updated:", cursor.rowcount)
print("Current customer state:")
display(result_df)


Rows inserted or updated: 0
Current customer state:


,customer_id,city,event_time,record_id
0,C001,Bengaluru,2026-10-25 11:00:00,202


In [14]:

# Step 11C: UPSERT a newer customer event

# New customer version
newer_event = (
    "C001",
    "Pune",
    "2026-10-25 12:00:00",
    204
)

# Reuse the conditional UPSERT from Step 11B
cursor = conn.execute(upsert_sql, newer_event)

# Commit the database change
conn.commit()

# Verify the current-state table
updated_df = pd.read_sql_query(
    """
    SELECT *
    FROM customers_current
    ORDER BY customer_id;
    """,
    conn
)

print("Rows inserted or updated:", cursor.rowcount)
print("Current customer state:")
display(updated_df)


Rows inserted or updated: 1
Current customer state:


,customer_id,city,event_time,record_id
0,C001,Pune,2026-10-25 12:00:00,204


In [15]:

# Step 11D: Insert a new customer using UPSERT

# New customer event
new_customer_event = (
    "C002",
    "Mumbai",
    "2026-10-25 09:00:00",
    205
)

# Reuse the existing conditional UPSERT
cursor = conn.execute(
    upsert_sql,
    new_customer_event
)

# Commit the change
conn.commit()

# Verify the target table
current_state_df = pd.read_sql_query(
    """
    SELECT
        customer_id,
        city,
        event_time,
        record_id
    FROM customers_current
    ORDER BY customer_id;
    """,
    conn
)

print("Rows inserted or updated:", cursor.rowcount)
print("Current-state customer records:")
display(current_state_df)

print("Total current customers:", len(current_state_df))


Rows inserted or updated: 1
Current-state customer records:


,customer_id,city,event_time,record_id
0,C001,Pune,2026-10-25 12:00:00,204
1,C002,Mumbai,2026-10-25 09:00:00,205


Total current customers: 2


In [16]:

# Step 11E: Test UPSERT idempotency

# Read target state before replay
before_df = pd.read_sql_query(
    """
    SELECT *
    FROM customers_current
    ORDER BY customer_id;
    """,
    conn
)

# Replay the exact same C002 event
replayed_event = (
    "C002",
    "Mumbai",
    "2026-10-25 09:00:00",
    205
)

# Reuse our existing conditional UPSERT
cursor = conn.execute(
    upsert_sql,
    replayed_event
)

conn.commit()

# Read target state after replay
after_df = pd.read_sql_query(
    """
    SELECT *
    FROM customers_current
    ORDER BY customer_id;
    """,
    conn
)

# Validate whether target state changed
print("Rows inserted or updated:", cursor.rowcount)

print("Target unchanged:",
      before_df.equals(after_df))

print("Total current customers:", len(after_df))

display(after_df)


Rows inserted or updated: 0
Target unchanged: True
Total current customers: 2


,customer_id,city,event_time,record_id
0,C001,Pune,2026-10-25 12:00:00,204
1,C002,Mumbai,2026-10-25 09:00:00,205


In [17]:

# Step 12: Final current-state quality checks

dq_query = """
WITH metrics AS (
    SELECT
        COUNT(*) AS total_records,

        COUNT(DISTINCT customer_id)
            AS unique_customers,

        SUM(
            CASE
                WHEN customer_id IS NULL
                  OR TRIM(customer_id) = ''
                THEN 1 ELSE 0
            END
        ) AS invalid_customer_ids,

        SUM(
            CASE
                WHEN city IS NULL
                  OR TRIM(city) = ''
                  OR event_time IS NULL
                  OR record_id IS NULL
                THEN 1 ELSE 0
            END
        ) AS invalid_attributes

    FROM customers_current
)
SELECT
    *,
    CASE
        WHEN total_records = unique_customers
         AND invalid_customer_ids = 0
         AND invalid_attributes = 0
         AND total_records = 2
        THEN 'PASS'
        ELSE 'FAIL'
    END AS dq_status
FROM metrics;
"""

dq_result = pd.read_sql_query(dq_query, conn)

print("Current-state Data Quality results:")
display(dq_result)


Current-state Data Quality results:


,total_records,unique_customers,invalid_customer_ids,invalid_attributes,dq_status
0,2,2,0,0,PASS


In [18]:

# Step 13: Reconcile observed UPSERT outcomes

audit_query = """
WITH event_audit AS (
    SELECT 1 AS event_attempt,
           'C001' AS customer_id,
           'IGNORED_OLDER' AS outcome

    UNION ALL

    SELECT 2, 'C001', 'UPDATED'

    UNION ALL

    SELECT 3, 'C002', 'INSERTED'

    UNION ALL

    SELECT 4, 'C002', 'IGNORED_REPLAY'
),
audit_metrics AS (
    SELECT
        COUNT(*) AS total_attempts,

        SUM(CASE
            WHEN outcome = 'INSERTED' THEN 1
            ELSE 0
        END) AS inserts,

        SUM(CASE
            WHEN outcome = 'UPDATED' THEN 1
            ELSE 0
        END) AS updates,

        SUM(CASE
            WHEN outcome IN (
                'IGNORED_OLDER',
                'IGNORED_REPLAY'
            ) THEN 1
            ELSE 0
        END) AS ignored

    FROM event_audit
),
target_metrics AS (
    SELECT
        COUNT(*) AS target_count,
        COUNT(DISTINCT customer_id)
            AS unique_customers
    FROM customers_current
)
SELECT
    a.*,
    t.target_count,
    t.unique_customers,

    CASE
        WHEN a.total_attempts =
             a.inserts + a.updates + a.ignored
         AND t.target_count = 2
         AND t.unique_customers = t.target_count
        THEN 'PASS'
        ELSE 'FAIL'
    END AS audit_status

FROM audit_metrics a
CROSS JOIN target_metrics t;
"""

audit_df = pd.read_sql_query(
    audit_query,
    conn
)

print("Final CDC audit reconciliation:")
display(audit_df)


Final CDC audit reconciliation:


,total_attempts,inserts,updates,ignored,target_count,unique_customers,audit_status
0,4,1,1,2,2,2,PASS


In [19]:

# Step 14: Value-level reconciliation

validation_sql = """
WITH expected AS (
    SELECT
        'C001' AS customer_id,
        'Pune' AS city,
        '2026-10-25 12:00:00' AS event_time,
        204 AS record_id

    UNION ALL

    SELECT
        'C002',
        'Mumbai',
        '2026-10-25 09:00:00',
        205
),
comparison AS (
    SELECT
        e.customer_id,
        e.city AS expected_city,
        c.city AS actual_city,
        e.event_time AS expected_time,
        c.event_time AS actual_time,
        e.record_id AS expected_record_id,
        c.record_id AS actual_record_id,

        CASE
            WHEN c.customer_id IS NULL
                THEN 'MISSING_TARGET'
            WHEN e.city IS c.city
             AND e.event_time IS c.event_time
             AND e.record_id IS c.record_id
                THEN 'MATCH'
            ELSE 'VALUE_MISMATCH'
        END AS validation_status

    FROM expected e
    LEFT JOIN customers_current c
        ON e.customer_id = c.customer_id
)
SELECT *
FROM comparison
ORDER BY customer_id;
"""

value_check_df = pd.read_sql_query(
    validation_sql,
    conn
)

display(value_check_df)

print("Matched customers:",
      (value_check_df["validation_status"] == "MATCH").sum())

print("Mismatched customers:",
      (value_check_df["validation_status"] != "MATCH").sum())


,customer_id,expected_city,actual_city,expected_time,actual_time,expected_record_id,actual_record_id,validation_status
0,C001,Pune,Pune,2026-10-25 12:00:00,2026-10-25 12:00:00,204,204,MATCH
1,C002,Mumbai,Mumbai,2026-10-25 09:00:00,2026-10-25 09:00:00,205,205,MATCH


Matched customers: 2
Mismatched customers: 0


In [20]:

from pathlib import Path

print("Working directory:", Path.cwd())

print("CSV exists:", Path("customers.csv").is_file())

print("Database exists:", Path("customers.db").is_file())


Working directory: /home/itv028683
CSV exists: False
Database exists: False


In [21]:

from pathlib import Path

project_folder = Path(
    "/home/itv028683/question_02_deduplication"
)

%cd /home/itv028683/question_02_deduplication

print("Working directory:", Path.cwd())
print("CSV exists:", Path("customers.csv").is_file())
print("Database exists:", Path("customers.db").is_file())


/home/itv028683/question_02_deduplication
Working directory: /home/itv028683/question_02_deduplication
CSV exists: True
Database exists: True


In [22]:

import sqlite3
import pandas as pd

# Connect to the existing project database
conn = sqlite3.connect("customers.db")

# List available SQLite tables
tables_df = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    conn
)

print("Tables in customers.db:")
display(tables_df)


Tables in customers.db:


,name
0,customers_current
1,customers_landing


In [23]:

# Step 15K: Verify landing and current-state records

landing_df = pd.read_sql_query(
    "SELECT * FROM customers_landing ORDER BY record_id;",
    conn
)

current_df = pd.read_sql_query(
    "SELECT * FROM customers_current ORDER BY customer_id;",
    conn
)

print("LANDING TABLE:")
display(landing_df)

print("CURRENT-STATE TABLE:")
display(current_df)

print("Landing count:", len(landing_df))
print("Current-state count:", len(current_df))


LANDING TABLE:


,record_id,customer_id,city,updated_at
0,101,C001,Hyderabad,2026-10-20
1,102,C001,Chennai,2026-10-22
2,103,C001,Bengaluru,2026-10-24
3,104,C002,Mumbai,2026-10-23


CURRENT-STATE TABLE:


,customer_id,city,event_time,record_id
0,C001,Pune,2026-10-25 12:00:00,204
1,C002,Mumbai,2026-10-25 09:00:00,205


Landing count: 4
Current-state count: 2
